In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from collections import defaultdict
import ast


In [2]:
df = pd.read_csv('/Users/oskarkarlsson/Desktop/DTU/4. Semester/Social Science/CompSci/Week4/final_papers.csv')
df['author_ids'] = df['author_ids'].apply(ast.literal_eval)
df

,id,publication_year,cited_by_count,author_ids
0,https://openalex.org/W2115022330,2007.0,3050,"[https://openalex.org/A5083710366, https://ope..."
1,https://openalex.org/W2522448907,2016.0,2339,"[https://openalex.org/A5016636016, https://ope..."
2,https://openalex.org/W1586856071,1994.0,1269,"[https://openalex.org/A5073802250, https://ope..."
3,https://openalex.org/W2171057921,2000.0,935,"[https://openalex.org/A5030296019, https://ope..."
4,https://openalex.org/W2113920515,2000.0,416,"[https://openalex.org/A5073802250, https://ope..."
...,...,...,...,...
215253,https://openalex.org/W2971743178,2019.0,11,"[https://openalex.org/A5112531677, https://ope..."
215254,https://openalex.org/W3008208901,2020.0,26,"[https://openalex.org/A5101891152, https://ope..."
215255,https://openalex.org/W2552329007,2016.0,24,"[https://openalex.org/A5009725517, https://ope..."
215256,https://openalex.org/W3083515752,2020.0,21,[https://openalex.org/A5065423139]


### Create the graph

In [5]:
G = nx.Graph() # Create an empty graph

pair_counts = defaultdict(int)


for author_list in df['author_ids']: # Iterate through each paper's author list
    author_list = [a for a in author_list if a is not None]
    for i in range(len(author_list)): # Iterate through each author in the list
        for j in range(i + 1, len(author_list)): # Iterate through pairs of authors
            pair = tuple(sorted([author_list[i].strip(), author_list[j].strip()])) # Create a sorted tuple of unique author pairs
            pair_counts[pair] += 1 # Increment the count for this pair

# Convert the pair counts into a list of weighted edges for the graph
weighted_edgelist = [(a, b, count) for (a, b), count in pair_counts.items()]

# Create weighted edges in the graph
G.add_weighted_edges_from(weighted_edgelist) # Adds edges with weights in the metadata of the graph, also creates nodes
weights = [G[u][v]['weight'] for u, v in G.edges()] # Extract the weights for visualization

### Visualize the graph

In [6]:
import numpy as np
from matplotlib.collections import LineCollection

# 1. Use spectral layout as a fast starting position, then refine with fewer spring iterations
pos = nx.spectral_layout(G)
pos = nx.spring_layout(G, k=0.5, seed=42, pos=pos, iterations=15)  # far fewer iterations needed

# 2. Filter weak edges (only draw edges where weight >= 2)
min_weight = 2
filtered_edges = [(u, v) for u, v, d in G.edges(data=True) if d['weight'] >= min_weight]

# 3. Draw edges using LineCollection (much faster than draw_networkx_edges)
segments = [[pos[u], pos[v]] for u, v in filtered_edges]
lc = LineCollection(segments, colors='gray', alpha=0.3, linewidths=0.5)

fig, ax = plt.subplots(figsize=(14, 10))
ax.add_collection(lc)

# 4. Draw nodes using scatter (much faster than draw_networkx_nodes)
xs, ys = zip(*[pos[n] for n in G.nodes()])
ax.scatter(xs, ys, s=10, c='blue', alpha=0.7, zorder=2)

ax.autoscale()
ax.set_title('Co-authorship Network')
ax.axis('off')
plt.show()

KeyboardInterrupt: 

### Add attributes to nodes

In [ ]:
for node in G.nodes():
    nx.set_node_attributes(G, {node: G.degree(node)}, 'degree') # change to correct information